# 🛡️ Glu-Stock: 00b_MODEL_RETRAINING_CNN
**Phase**: Dynamic Cross-Sectional Intelligence (CNN Brain)

This notebook trains the CNN 'Super Brain' on a dynamic panel dataset containing 5 years of historical data from the latest active LQ45 constituents. It scrapes the current LQ45 members to ensure the deep learning model spots structural patterns on highly liquid assets.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas tensorflow lxml html5lib

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE & DYNAMIC DATA
import json, os, firebase_admin, numpy as np, pandas as pd, yfinance as yf
import tensorflow as tf
from firebase_admin import credentials, db
from datetime import datetime
from kaggle_secrets import UserSecretsClient

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON"))
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")
        
    def log_event(self, phase, details):
        self.root_ref.child("history").push({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        tables = pd.read_html('https://id.wikipedia.org/wiki/LQ45')
        for df in tables:
            if 'Kode' in df.columns:
                tickers = (df['Kode'] + '.JK').tolist()
                print("✅ Dynamically fetched LQ45 list.")
                return tickers
    except Exception as e:
        print("⚠️ Wikipedia fetch failed. Using fallback LQ45 list.")
    return fallback

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Panel Training Pipeline)
def build_cnn_panel_data(universe, seq_len=30, period="5y"):
    all_X, all_y = [], []
    print(f"📉 Fetching {period} of data for {len(universe)} tickers...")
    for ticker in universe:
        df = yf.download(ticker, period=period, progress=False)
        if len(df) > seq_len:
            data = df[['Open', 'High', 'Low', 'Close', 'Volume']].values
            # Normalize per ticker to allow cross-sectional merging
            data = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0) + 1e-7)
            for i in range(len(data) - seq_len - 1):
                all_X.append(data[i:i+seq_len])
                # 1 if price goes up tomorrow, else 0
                y_val = 1 if df['Close'].iloc[i+seq_len+1] > df['Close'].iloc[i+seq_len] else 0
                all_y.append(y_val)
    print("✅ Successfully aggregated market sequences.")
    return np.array(all_X), np.array(all_y)

def train_cnn(X, y):
    print(f"🧠 Training CNN Super Brain on {len(y)} target sequences...")
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(32, 3, activation='relu', input_shape=(30, 5)),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X, y, epochs=10, batch_size=64, validation_split=0.2, verbose=1)
    return model

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = "/kaggle/working/"
    
    universe = get_dynamic_lq45()
    
    # 1. Build Panel & Train CNN
    X_train, y_train = build_cnn_panel_data(universe)
    cnn_model = train_cnn(X_train, y_train)
    
    print("⚙️ Converting to TFLite...")
    converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
    tflite_model = converter.convert()
    with open(os.path.join(output_dir, "cnn_daily_t2.tflite"), "wb") as f:
        f.write(tflite_model)
        
    fb.log_event("RETRAINING_CNN", f"Completed CNN panel update on {len(universe)} dynamic LQ45 tickers.")
    print(f"✅ CNN Model updated successfully in WORKING directory with {len(y_train)} sequences.")

run_retrain()